# wandb-config-into-args — worked example 3: Merge sweep config while preserving non-hparam fields

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `wandb-config-into-args`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

The `hasattr`-based update pattern is non-destructive toward fields that are absent from the sweep config. Fields like `wandb_project`, `wandb_name`, or `device` that the sweep never samples are simply left at their dataclass defaults. This makes the update safe to apply even when the sweep config is a strict subset of the args fields.

## Worked solution

**Step 1 — partial vs full override.**
A sweep config typically samples only the hyperparameters being tuned (e.g., `lr`, `batch_size`, `weight_decay`). Other args fields (`wandb_project`, `device`, `n_epochs`) are not in the sweep config and must not be touched.

**Step 2 — hasattr as a set-intersection filter.**
When we check `if hasattr(args, k)`, we are effectively computing the intersection of the sweep config's keys with the args dataclass's fields. Fields that are in the dataclass but not in the config are untouched. Fields that are in the config but not in the dataclass are skipped.

**Step 3 — verify with a side-by-side print.**
After applying the update, we print the original defaults (captured with `dataclasses.asdict`) and the updated values side-by-side to confirm that only sampled fields changed.

In [ ]:
import sys
from unittest.mock import MagicMock
from dataclasses import dataclass, asdict

sys.modules.setdefault('wandb', MagicMock())

@dataclass
class FullArgs:
    lr: float = 1e-3
    batch_size: int = 32
    weight_decay: float = 0.0
    n_epochs: int = 10
    device: str = 'cpu'
    wandb_project: str = 'my-exp'
    wandb_name: str = 'run-1'

def apply_partial_sweep(args, sweep_cfg):
    """Apply sweep config (a subset of args fields) to args."""
    for k, v in sweep_cfg.items():
        if hasattr(args, k):
            setattr(args, k, v)
    return args

# Exercise it
args = FullArgs()
defaults_snapshot = asdict(args).copy()
print('Defaults:', defaults_snapshot)

# Sweep only varies lr and weight_decay
sweep_cfg = {'lr': 1e-4, 'weight_decay': 0.05, '_wandb': {}}
apply_partial_sweep(args, sweep_cfg)

print('After sweep:', asdict(args))
print('lr changed:', args.lr)          # 1e-4
print('device unchanged:', args.device)  # still 'cpu'
print('project unchanged:', args.wandb_project)  # still 'my-exp'